In [ ]:
# =========================
# SMILES Tokenizer (robust) + build vocab + encode to IDs + save
# Colab/Drive friendly
# بقلك هاد كود مش سلف اوير سمايل Vocab-Token
# - Fixes the "Nc / Cc" issue by allowing ONLY known 2-letter elements
# - Keeps bracket atoms like [nH], [N+], [O-] as single tokens
# =========================

from google.colab import drive
drive.mount('/content/drive')

import re
import pandas as pd
import numpy as np
from collections import Counter

# ---- 1) paths ----
input_csv  = "/content/drive/MyDrive/Diss/smiles.csv"
smiles_col = "smiles"
out_dir    = "/content/drive/MyDrive/Diss/"

# ---- 2) robust SMILES tokenizer (fixed) ----
# Key fix:
# - we DO NOT use [A-Z][a-z]? because it incorrectly merges "Nc", "Cc", ...
# - instead: allow ONLY a curated list of 2-letter elements (Cl, Br, Si, Na, ...)
# - and otherwise single-letter elements [A-Z]

TWO_CHAR_ELEMENTS = (
    "Br|Cl|Si|Na|Ca|Li|Al|Mg|Zn|Fe|As|Se|Ag|Au|Sn|Hg|Cu|Mn|Co|Ni|Cr|Pt|Pd|Pb|Bi|Sb|Te"
)

SMILES_TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+\]"              # bracket atom: [nH], [N+], [O-], ...
    r"|" + TWO_CHAR_ELEMENTS +  # known 2-letter elements only
    r"|\%\d{2}"                 # ring numbers like %10
    r"|\d"                      # ring numbers 0-9
    r"|\(|\)"                   # branches
    r"|\.|=|#|-|\+|\\|/|:"       # bonds and dot
    r"|@@?|H"                   # chirality @ @@, explicit H
    r"|[A-Z]"                   # single-letter elements: C N O S P F I B ...
    r"|[bcnops]"                # aromatic
    r")"
)

def tokenize_smiles(smiles: str):
    if smiles is None:
        return []
    s = str(smiles).strip()
    if not s:
        return []
    tokens = SMILES_TOKEN_PATTERN.findall(s)

    # sanity check: if tokenization doesn't cover whole string, mark as failed
    if "".join(tokens) != s:
        return ["<BAD_TOKENIZE>"] + list(s)
    return tokens

# ---- 3) load data ----
df = pd.read_csv(input_csv)
df[smiles_col] = df[smiles_col].astype(str).fillna("").str.strip()

# optional: drop empty
df = df[df[smiles_col].str.len() > 0].reset_index(drop=True)

# tokenize all
token_lists = df[smiles_col].apply(tokenize_smiles).tolist()

# quick report
bad_tok = sum(1 for t in token_lists if len(t) > 0 and t[0] == "<BAD_TOKENIZE>")
print("Total SMILES:", len(token_lists))
print("Bad tokenizations:", bad_tok)

# show few examples
for i in range(min(5, len(token_lists))):
    print("\nSMILES:", df.loc[i, smiles_col])
    print("TOKENS:", token_lists[i])

# ---- 4) build vocab ----
PAD = "<pad>"
BOS = "<bos>"
EOS = "<eos>"

counter = Counter(tok for toks in token_lists for tok in toks)

# put special tokens first
all_tokens = [PAD, BOS, EOS] + sorted(counter.keys())

token2idx = {tok: i for i, tok in enumerate(all_tokens)}
idx2token = {i: tok for tok, i in token2idx.items()}

pad_id = token2idx[PAD]
bos_id = token2idx[BOS]
eos_id = token2idx[EOS]

print("\nVocab size:", len(token2idx))
print("Special IDs:", {"PAD": pad_id, "BOS": bos_id, "EOS": eos_id})

# ---- 5) encode to ids + add BOS/EOS + pad to max_len ----
encoded = []
lengths = []

for toks in token_lists:
    ids = [bos_id] + [token2idx.get(t, token2idx.get("<BAD_TOKENIZE>", pad_id)) for t in toks] + [eos_id]
    encoded.append(ids)
    lengths.append(len(ids))

max_len = int(np.percentile(lengths, 99))  # padding length (99th percentile)
max_len = max(max_len, 8)                  # minimum
print("Max len (99th percentile):", max_len)

X = np.full((len(encoded), max_len), pad_id, dtype=np.int64)
for i, ids in enumerate(encoded):
    ids = ids[:max_len]          # truncate
    X[i, :len(ids)] = ids

# ---- 6) save outputs ----
tokenized_path = out_dir + "tokenized_smiles_new.csv"
pd.DataFrame(X).to_csv(tokenized_path, index=False)

vocab_path = out_dir + "vocab_new.csv"
vocab_df = pd.DataFrame({"index": list(idx2token.keys()), "token": list(idx2token.values())})
vocab_df = vocab_df.sort_values("index")
vocab_df.to_csv(vocab_path, index=False)

preview_path = out_dir + "tokenized_smiles_tokens_preview.csv"
preview_df = df[[smiles_col]].copy()
preview_df["tokens"] = token_lists
preview_df["len_tokens"] = [len(t) for t in token_lists]
preview_df.to_csv(preview_path, index=False)

print("\n✅ Saved:")
print(" -", tokenized_path)
print(" -", vocab_path)
print(" -", preview_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total SMILES: 1591378
Bad tokenizations: 0

SMILES: CCCC(=O)NNC(=O)Nc1ccccc1
TOKENS: ['C', 'C', 'C', 'C', '(', '=', 'O', ')', 'N', 'N', 'C', '(', '=', 'O', ')', 'N', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1']

SMILES: CC(=O)NC1CCC2(C)C(CCC3(C)C2C(=O)C=C2C4C(C)C(C)CCC4(C)CCC23C)C1(C)C(=O)O
TOKENS: ['C', 'C', '(', '=', 'O', ')', 'N', 'C', '1', 'C', 'C', 'C', '2', '(', 'C', ')', 'C', '(', 'C', 'C', 'C', '3', '(', 'C', ')', 'C', '2', 'C', '(', '=', 'O', ')', 'C', '=', 'C', '2', 'C', '4', 'C', '(', 'C', ')', 'C', '(', 'C', ')', 'C', 'C', 'C', '4', '(', 'C', ')', 'C', 'C', 'C', '2', '3', 'C', ')', 'C', '1', '(', 'C', ')', 'C', '(', '=', 'O', ')', 'O']

SMILES: CC(=O)NC(C)Cc1ccc(C#Cc2ccnc(N3CCCC(F)C3)n2)cc1
TOKENS: ['C', 'C', '(', '=', 'O', ')', 'N', 'C', '(', 'C', ')', 'C', 'c', '1', 'c', 'c', 'c', '(', 'C', '#', 'C', 'c', '2', 'c', 'c', 'n', 'c', '(', 'N', '3', 'C', 

In [1]:
# =========================
# SMILES -> Tokenize + build vocab + encode to IDs + save بقلك هاد سلف اوير سمايلز وتوكنيزيشن
# Outputs:
# - tokenized_smiles_new.csv
# - vocab_smiles_new.csv
# - smiles_tokens_preview.csv
# =========================

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import pandas as pd
import numpy as np
from collections import Counter

# ---- 1) paths ----
input_csv  = "/content/drive/MyDrive/Diss/smiles.csv"
smiles_col = "smiles"
out_dir    = "/content/drive/MyDrive/Diss/"

# ---- 2) load data ----
df = pd.read_csv(input_csv)
df[smiles_col] = df[smiles_col].astype(str).fillna("").str.strip()
df = df[df[smiles_col].str.len() > 0].reset_index(drop=True)

print("Loaded rows:", len(df))

# ---- 3) SMILES tokenizer ----
# Tokenizer مناسب وبسيط للـ SMILES
SMILES_TOKEN_PATTERN = r"(\%\d{2}|Br|Cl|\[[^\]]+\]|Si|Se|Na|Li|Ca|@@?|=|#|-|\+|\\|/|\(|\)|\.|:|~|@|\?|>>?|\*|\$|\d|[A-Za-z])"

token_regex = re.compile(SMILES_TOKEN_PATTERN)

def tokenize_smiles(smiles: str):
    s = str(smiles).strip()
    if not s:
        return []
    tokens = token_regex.findall(s)
    reconstructed = "".join(tokens)
    if reconstructed != s:
        # fallback: character-level if regex misses something
        return ["<BAD_TOKENIZE>"] + list(s)
    return tokens

token_lists = df[smiles_col].apply(tokenize_smiles).tolist()

bad_tok = sum(1 for t in token_lists if len(t) > 0 and t[0] == "<BAD_TOKENIZE>")
print("Bad tokenizations:", bad_tok)

print("\nExamples:")
for i in range(min(5, len(df))):
    print("SMILES :", df.loc[i, smiles_col])
    print("TOKENS :", token_lists[i])
    print("---")

# ---- 4) build vocab ----
PAD = "<pad>"
BOS = "<bos>"
EOS = "<eos>"

counter = Counter(tok for toks in token_lists for tok in toks)

all_tokens = [PAD, BOS, EOS] + sorted(counter.keys())
token2idx = {tok: i for i, tok in enumerate(all_tokens)}
idx2token = {i: tok for tok, i in token2idx.items()}

pad_id = token2idx[PAD]
bos_id = token2idx[BOS]
eos_id = token2idx[EOS]

print("\nVocab size:", len(token2idx))
print("Special IDs:", {"PAD": pad_id, "BOS": bos_id, "EOS": eos_id})

# ---- 5) encode to IDs + add BOS/EOS + pad ----
encoded = []
lengths = []
bad_id = token2idx.get("<BAD_TOKENIZE>", pad_id)

for toks in token_lists:
    ids = [bos_id] + [token2idx.get(t, bad_id) for t in toks] + [eos_id]
    encoded.append(ids)
    lengths.append(len(ids))

max_len = int(np.percentile(lengths, 99))
max_len = max(max_len, 8)

print("Max len (99th percentile):", max_len)

X = np.full((len(encoded), max_len), pad_id, dtype=np.int64)
for i, ids in enumerate(encoded):
    ids = ids[:max_len]
    X[i, :len(ids)] = ids

# ---- 6) save outputs ----
tokenized_path = os.path.join(out_dir, "tokenized_smiles_new.csv")
pd.DataFrame(X).to_csv(tokenized_path, index=False)

vocab_path = os.path.join(out_dir, "vocab_smiles_new.csv")
vocab_df = pd.DataFrame({
    "index": list(idx2token.keys()),
    "token": list(idx2token.values())
}).sort_values("index")
vocab_df.to_csv(vocab_path, index=False)

preview_path = os.path.join(out_dir, "smiles_tokens_preview.csv")
preview_df = df[[smiles_col]].copy()
preview_df["tokens"] = token_lists
preview_df["len_tokens"] = [len(t) for t in token_lists]
preview_df.to_csv(preview_path, index=False)

print("\n✅ Saved:")
print(" -", tokenized_path)
print(" -", vocab_path)
print(" -", preview_path)

Mounted at /content/drive
Loaded rows: 1591378
Bad tokenizations: 0

Examples:
SMILES : CCCC(=O)NNC(=O)Nc1ccccc1
TOKENS : ['C', 'C', 'C', 'C', '(', '=', 'O', ')', 'N', 'N', 'C', '(', '=', 'O', ')', 'N', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1']
---
SMILES : CC(=O)NC1CCC2(C)C(CCC3(C)C2C(=O)C=C2C4C(C)C(C)CCC4(C)CCC23C)C1(C)C(=O)O
TOKENS : ['C', 'C', '(', '=', 'O', ')', 'N', 'C', '1', 'C', 'C', 'C', '2', '(', 'C', ')', 'C', '(', 'C', 'C', 'C', '3', '(', 'C', ')', 'C', '2', 'C', '(', '=', 'O', ')', 'C', '=', 'C', '2', 'C', '4', 'C', '(', 'C', ')', 'C', '(', 'C', ')', 'C', 'C', 'C', '4', '(', 'C', ')', 'C', 'C', 'C', '2', '3', 'C', ')', 'C', '1', '(', 'C', ')', 'C', '(', '=', 'O', ')', 'O']
---
SMILES : CC(=O)NC(C)Cc1ccc(C#Cc2ccnc(N3CCCC(F)C3)n2)cc1
TOKENS : ['C', 'C', '(', '=', 'O', ')', 'N', 'C', '(', 'C', ')', 'C', 'c', '1', 'c', 'c', 'c', '(', 'C', '#', 'C', 'c', '2', 'c', 'c', 'n', 'c', '(', 'N', '3', 'C', 'C', 'C', 'C', '(', 'F', ')', 'C', '3', ')', 'n', '2', ')', 'c', 'c', '1']
---
SMI